# Bank Account Fraud Detection · Reproducible Experiment

**Ilia Jakhaia** · BAF NeurIPS 2022 (`Base.csv`) · forward-in-time validation and review capacity.

This notebook is a readable interface to the same source code used by `baf-fraud`. It has **no prepopulated results**: rerun it after obtaining the official dataset. The README's historical baseline metrics come from a **separate, previously executed** notebook, stored in `results/reference_notebook_results.json`. The new sentinel experiment's results have **not** been verified on the full dataset.

> Run from the repository root after `python -m pip install -e ".[notebook]"`. The local dataset is intentionally excluded from GitHub.

## 1. Data loading and split

Data has 1 million synthetic applications and eight index months (0–7); months are **not calendar dates**. Months 0–5 are training, month 6 is validation, and month 7 is final test. The target and time index are excluded from predictor inputs.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from baf_fraud.data import load_base, chronological_split
from baf_fraud.run import run_experiment

csv = Path("data/Base.csv")
if not csv.is_file():
    raise FileNotFoundError("Download Base.csv into data/; see data/README.md")
df = load_base(csv)
print(f"Dataset: {len(df):,} rows × {df.shape[1]} columns")
display(df.groupby("month")["fraud_bool"].agg(applications="size", frauds="sum", fraud_rate="mean"))

In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = chronological_split(df)
for name, labels in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name}: {len(labels):,} applications; fraud prevalence {labels.mean():.3%}")
assert "fraud_bool" not in X_train.columns and "month" not in X_train.columns

## 2. One-pass model selection and final held-out evaluation

The source pipeline compares two original baselines and two sentinel-aware logistic regression variants (`C=0.3`, `C=1.0`). **All** hyperparameter and model choices are made on validation month 6 using average precision (and Precision@1% as a tie-breaker). Preprocessing is learned on training months 0–5. Only the selected model is evaluated on test month 7.

This step fits **four models on the full training partition** and may require significant RAM and CPU time. It exports `results/generated/metrics.json` and charts. No results from this experiment are prefilled.

In [ ]:
report = run_experiment(df, output="results/generated", capacity=0.01)
validation_table = pd.DataFrame(report["validation"]).T
cols = ["average_precision", "precision_at_capacity", "recall_at_capacity", "frauds_detected"]
display(validation_table[cols].sort_values("average_precision", ascending=False))
print("Chosen on validation:", report["selected_model"])
print("Test metrics:")
display(pd.Series(report["test"]))

## 3. Operational investigation-budget sensitivity

Compare what happens if the bank reviews the riskiest 0.25%, 0.5%, 1%, 2% or 5% of applications. These figures are derived from the **selected model's actual scores**; they are not calculated from historical summary statistics. The fixed 0.5 score threshold and the top-1% review policy are different operating points.

In [ ]:
display(pd.DataFrame(report["validation_budget_sweep"][report["selected_model"]]).T)
display(pd.DataFrame(report["test_budget_sweep"]).T)
for name in ["validation_model_comparison", "validation_budget_sweep", "test_budget_sweep", "test_precision_recall_curve", "test_confusion_matrix"]:
    display(Image(filename=f"results/generated/figures/{name}.png"))

## 4. Interpretation and cautions

- **Average precision** measures rare-event ranking quality; it is not the same as precision at a fixed 0.5 threshold.
- **Precision@1%** is the fraction of actual fraud in the manually reviewed top-scored 1% of applications. **Recall@1%** is the fraction of all fraud in the split surfaced by that review list.
- The sentinel experiment converts exact `-1` only in selected columns and retains missingness flags. Some other negative numbers are valid values and are not changed.
- Synthetic BAF data do not establish real-world economic benefits, fairness, or calibration. A single held-out month is not sufficient to claim production readiness.

See `docs/EXPERIMENT_PROTOCOL.md` for the full comparison contract and threats to validity.